## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-25-22-54-10 +0000,nypost,Trump’s DC renovation account gets mysterious ...,https://nypost.com/2026/11/25/us-news/presiden...
1,2026-11-25-22-53-38 +0000,nypost,Hit-and-run driver busted 3 months after alleg...,https://nypost.com/2026/11/25/us-news/hit-and-...
2,2026-11-25-22-52-05 +0000,nypost,Ticketmaster bosses rake in $54M in wake of il...,https://nypost.com/2026/11/25/business/ticketm...
3,2026-11-25-22-49-04 +0000,bbc,Angelina Jolie and Brad Pitt settle legal batt...,https://www.bbc.co.uk/news/articles/cmvg2989wv...
4,2026-11-25-22-42-49 +0000,wapo,Macy’s Thanksgiving Day Parade postponed by st...,https://www.washingtonpost.com/nation/2026/11/...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2791/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
0,trump,33
85,new,22
343,china,20
122,nvidia,20
156,ai,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
395,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...,129
275,2026-11-25-11-58-00 +0000,wsj,Nvidia’s Jensen Huang is set for at least two ...,https://www.wsj.com/politics/policy/trump-says...,118
212,2026-11-25-15-46-00 +0000,wsj,There was no evidence a Chinese tech executive...,https://www.wsj.com/politics/national-security...,107
171,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...,106
204,2026-11-25-15-58-36 +0000,nyt,Trump Administration Unveils New Tariffs for C...,https://www.nytimes.com/2026/11/25/business/tr...,103


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
395,129,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...
171,86,2026-11-25-17-24-25 +0000,wapo,"Trump proposes new tariffs on China, Europe an...",https://www.washingtonpost.com/politics/2026/1...
91,65,2026-11-25-20-41-58 +0000,nypost,Macy’s Thanksgiving Day Parade postponed due t...,https://nypost.com/2026/11/25/us-news/macys-th...
97,59,2026-11-25-20-27-58 +0000,nypost,Iowa lead explosion’s toxic dust could drift u...,https://nypost.com/2026/11/25/us-news/toxic-le...
366,55,2026-11-25-02-00-00 +0000,wsj,What would happen to the U.S. economy if the A...,https://www.wsj.com/economy/worse-than-the-dot...
262,49,2026-11-25-12-53-36 +0000,nypost,Kilmar Abrego García lands in Spain after self...,https://nypost.com/2026/11/25/world-news/kilma...
249,49,2026-11-25-13-49-04 +0000,cbc,Libyan army says 2 officers were killed in Ukr...,https://www.cbc.ca/news/world/ukraine-russia-l...
358,47,2026-11-25-03-00-00 +0000,wsj,A Democratic push to enact Medicare for All is...,https://www.wsj.com/politics/policy/democrats-...
396,43,2026-11-25-00-13-52 +0000,nypost,"Andrew Giuliani’s daughter, 4, steals spotligh...",https://nypost.com/2026/11/24/us-news/andrew-g...
372,41,2026-11-25-01-55-34 +0000,latimes,Thomas Markle indicted on child porn charges; ...,https://www.latimes.com/entertainment-arts/sto...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
